# Elvis Presley Sound Features Analysis - Interactive Visualization

## How to Interact with These Plots:
- **Click and drag** on the top scatter plot (Valence vs Acousticness) to select songs
- The **tempo histogram** in the middle will highlight selected songs
- The **bottom scatter plot** (Loudness vs Duration) will also highlight your selection
- All three charts are linked - your selection affects all visualizations
- **Click anywhere** on white space to clear your selection

## Time Signature Definition:
**Time signature** indicates the number of beats per measure in a song. Most songs are in 4/4 time (4 beats per measure), but Elvis also recorded songs in 3/4 time (waltz time, like 'Can't Help Falling in Love') and occasionally 5/4 time. The time signature affects the rhythmic feel and structure of the music.

In [ ]:
import pandas as pd
import altair as alt

In [ ]:
# Load Elvis dataset
df = pd.read_csv('dataset.csv')
elvis_df = df[df['artists'] == 'Elvis Presley'].reset_index(drop=True)

# Convert duration
elvis_df['duration_min'] = elvis_df['duration_ms'] / 60000

# Normalize loudness (scale clarity)
elvis_df['loudness_norm'] = (elvis_df['loudness'] - elvis_df['loudness'].min()) / \
                            (elvis_df['loudness'].max() - elvis_df['loudness'].min())

elvis_df.head()

In [ ]:
# Create a brush selection that will link all three charts
brush = alt.selection_interval(encodings=['x', 'y'])

In [ ]:
# Define green color scale for tempo
tempo_scale = alt.Scale(
    domain=[elvis_df['tempo'].min(), elvis_df['tempo'].max()],
    range=['#008000', '#32CD32']  # Green to Lime Green
)

# Valence vs Acousticness: emotional tone vs. instrumentation
points_valence = (
    alt.Chart(elvis_df)
    .mark_circle(size=90, opacity=0.7)
    .encode(
        x=alt.X('acousticness:Q', title='Acousticness (Instrumental Quality)'),
        y=alt.Y('valence:Q', title='Valence (Happiness)'),
        color=alt.condition(
            brush,
            alt.Color('tempo:Q', scale=tempo_scale, title='Tempo (BPM)'),
            alt.value('#CCCCCC')  # Gray for unselected
        ),
        tooltip=['track_name', 'album_name', 'tempo', 'loudness', 'duration_min', 'time_signature']
    )
    .add_params(brush)
    .properties(
        title='Elvis Presley Songs: Valence vs. Acousticness (Click and drag to select)',
        width=600,
        height=400
    )
)

In [ ]:
# Tempo Distribution linked to selection
# Base histogram showing all data in gray
tempo_hist_base = (
    alt.Chart(elvis_df)
    .mark_bar(opacity=0.3)
    .encode(
        x=alt.X('tempo:Q', bin=alt.Bin(maxbins=25), title='Tempo (BPM)'),
        y=alt.Y('count()', title='Number of Songs'),
        color=alt.value('#CCCCCC')  # Gray for all data
    )
)

# Overlay histogram showing only selected data
tempo_hist_selected = (
    alt.Chart(elvis_df)
    .mark_bar(opacity=0.8)
    .encode(
        x=alt.X('tempo:Q', bin=alt.Bin(maxbins=25), title='Tempo (BPM)'),
        y=alt.Y('count()', title='Number of Songs'),
        color=alt.value('#00A36C')  # Jade for selected
    )
    .transform_filter(brush)
)

# Layer the histograms
tempo_hist = (tempo_hist_base + tempo_hist_selected).properties(
    title='Distribution of Elvis Presley Song Tempos (Highlights selected songs)',
    width=600,
    height=300
)

In [ ]:
# Loudness vs Duration linked to selection
points_loudness = (
    alt.Chart(elvis_df)
    .mark_circle(size=90, opacity=0.7)
    .encode(
        x=alt.X('duration_min:Q', title='Song Duration (minutes)'),
        y=alt.Y('loudness:Q', title='Loudness (dB)'),
        color=alt.condition(
            brush,
            alt.value('#4CBB17'),  # Kelly Green for selected
            alt.value('#CCCCCC')   # Gray for unselected
        ),
        tooltip=['track_name', 'album_name', 'tempo', 'valence', 'time_signature']
    )
    .properties(
        title='Loudness vs Duration (Highlights selected songs from above)',
        width=600,
        height=400
    )
)

In [ ]:
# Combine all visuals into one dashboard
final_viz = points_valence & tempo_hist & points_loudness
final_viz

In [ ]:
# Save to HTML
final_viz.save('ep_sound_comparison_interactive.html')